In [1]:
# 逐步骤生成下一个token并考虑温度的影响

In [2]:
import sys
from transformers import GPT2LMHeadModel, GPT2Tokenizer, PreTrainedModel
from loguru import logger
import torch
import os

os.environ["HTTP_PROXY"] = "http://127.0.0.1:6382"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:6382"

logger.remove()
logger.add(sys.stdout, level="DEBUG", colorize=True)

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval();

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

首先定义控制逐步生成的class，并设置相关借口供后续调用。其中涉及到的概念如下：
+ softmax数学表达式如下，其特点是结果都在0到1之间，且加和为1。同时在保持单调性的同时，会放大各成员之间的差距。
$$
\operatorname{softmax}(z_i)=\frac{\exp(z_i)}{\sum_{j=1}^{V}\exp(z_j)}
$$
+ 带温度的softmax表达式如下，
$$
p_i(T)=\operatorname{softmax}\left(\frac{z_i}{T}\right)=\frac{\exp\left(z_i/T\right)}{\sum_{j=1}^{V}\exp\left(z_j/T\right)},\qquad T>0
$$
    - 当 $T=1$ 时，退化为普通 Softmax；
    - 当 $0<T<1$ 时，概率分布更加尖锐，更偏向高 logit 的 token；
    - 当 $T>1$ 时，概率分布更加平坦，低 logit 的 token 也获得更多概率；
    - 当 $T\to 0^+$ 时，概率质量趋向集中在最大 logit 对应的 token 上。
+ multinominal的名字来源于“多项分布”，用来按照概率进行抽取

In [3]:
class StepByStepGenerate:
    """
    逐个 token 生成文本，展示完整的预测过程。

    参数：
        model: 使用的模型
        tokenizer: 使用的tokenizer
        temperature: 温度，用于控制"创造力"。越高越随机，越低越确定
        max_token_num: 限制的最大生成token数，默认为50
    """

    def __init__(self, model: PreTrainedModel, tokenizer: GPT2Tokenizer, temperature: float,
                 max_token_num: int = 50) -> None:
        self._model: PreTrainedModel = model
        self._tokenizer: GPT2Tokenizer = tokenizer
        self._max_token_num: int = max_token_num
        self.temperature: float = temperature

    def generate(self, prompt: str) -> str:
        encoded_tokens = self._tokenizer(prompt, return_tensors="pt")
        input_ids: torch.Tensor = encoded_tokens["input_ids"]
        generated_text: str = prompt
        logger.info(
            f"Begin generate process, temperature = {self.temperature}, initial prompt = {prompt}, using non-greedy settings.")

        for step in range(self._max_token_num):
            logger.debug(f"{'=' * 10} Step {step + 1:2d} {'=' * 10}")
            logger.debug(f"prompt = {generated_text}")
            with torch.no_grad():
                generated_outputs = self._model(input_ids)
                next_token_logits: torch.Tensor = generated_outputs.logits[0, -1, :]

                scaled_logits: torch.Tensor = next_token_logits / self.temperature
                token_probabilities: torch.Tensor = torch.softmax(scaled_logits, dim=0)

                next_token_id: torch.Tensor = torch.multinomial(token_probabilities, 1)
                next_token: str | list[str] = self._tokenizer.decode(next_token_id[0])
                token_probability: float = token_probabilities[next_token_id[0]].item()
                logger.debug(f"next token : {next_token}, token probability : {token_probability:.2%}")

                top3_probs, top3_indices = torch.topk(token_probabilities, 3)
                alternatives: list[str] = [
                    f"'{self._tokenizer.decode(top3_indices[j])}' {top3_probs[j].item():.2%}"
                    for j in range(3)
                ]
                chosen_marker: str = " ✓" if next_token_id[0] in top3_indices else ""
                logger.debug(
                    f"Step {step + 1:2d}: choose '{next_token}' ({token_probability:.2%}){chosen_marker}"
                    f"| Top 3: {', '.join(alternatives)}"
                )

                input_ids = torch.cat([input_ids, next_token_id.unsqueeze(0)], dim=1)
                generated_text += next_token
                logger.debug(f"generated text = {generated_text}")

                if any(mark in next_token for mark in (".", "!", "?")):
                    break
        return generated_text

尝试对该类进行初始化和调用，先以温度为1的，普通softmax情况为例进行试验，这里打印全部debug信息来展示内部过程。

In [16]:
DEFAULT_PROMPT: str = "The meaning of Life is"
logger.remove()
logger.add(sys.stdout, level="DEBUG", colorize=True)
normal_generator = StepByStepGenerate(gpt2_model, gpt2_tokenizer, temperature=1, max_token_num=50)
rtn = normal_generator.generate(DEFAULT_PROMPT)
logger.info(rtn)

2026-08-04 12:47:37.083 | INFO     | __main__:generate:22 - Begin generate process, temperature = 1, initial prompt = The meaning of Life is, using non-greedy settings.
2026-08-04 12:47:37.084 | DEBUG    | __main__:generate:26 - ========== Step  1 ==========
2026-08-04 12:47:37.084 | DEBUG    | __main__:generate:27 - prompt = The meaning of Life is
2026-08-04 12:47:37.108 | DEBUG    | __main__:generate:38 - next token :  the, token probability : 6.54%
2026-08-04 12:47:37.109 | DEBUG    | __main__:generate:46 - Step  1: choose ' the' (6.54%) ✓| Top 3: ' a' 6.57%, ' the' 6.54%, ' to' 5.65%
2026-08-04 12:47:37.109 | DEBUG    | __main__:generate:53 - generated text = The meaning of Life is the
2026-08-04 12:47:37.110 | DEBUG    | __main__:generate:26 - ========== Step  2 ==========
2026-08-04 12:47:37.110 | DEBUG    | __main__:generate:27 - prompt = The meaning of Life is the
2026-08-04 12:47:37.127 | DEBUG    | __main__:generate:38 - next token :  most, token probability : 1.69%
2026-08-0

当温度在0到1之间时，各logits的分数被放大，差距也被拉大。根据softmax的性质，更大的差距意味着高概率的token更容易被选中，因此模型的回答确定性更高

In [15]:
logger.remove()
logger.add(sys.stdout, level="INFO", colorize=True)
cold_generator = StepByStepGenerate(gpt2_model, gpt2_tokenizer, temperature=0.3, max_token_num=50)
rtn = cold_generator.generate(DEFAULT_PROMPT)
logger.info(rtn)

2026-08-04 12:47:30.917 | INFO     | __main__:generate:22 - Begin generate process, temperature = 1, initial prompt = The meaning of Life is, using non-greedy settings.
2026-08-04 12:47:30.980 | INFO     | __main__:<module>:5 - The meaning of Life is far from clear.


反过来，温度大于1，logits的差距被缩小，模型更可能选中各类不同的token

In [11]:
logger.remove()
logger.add(sys.stdout, level="INFO", colorize=True)
hot_generator = StepByStepGenerate(gpt2_model, gpt2_tokenizer, temperature=3, max_token_num=50)
rtn = hot_generator.generate(DEFAULT_PROMPT)
logger.info(rtn)

2026-08-04 12:47:16.576 | INFO     | __main__:generate:22 - Begin generate process, temperature = 3, initial prompt = The meaning of Life is, using non-greedy settings.
2026-08-04 12:47:17.320 | INFO     | __main__:<module>:5 - The meaning of Life is loads more statisticalBI Refer bout bro stereotypical Wellington子 Salary computers Fall influences 233 101472 1967 raybuffer phony v PURasc freshlysign Improved framework understandably struct Tamidget followers AthrettOV fiction agreeable ."


最后用一组对比试验来量化不同温度下不同的概率分布：

In [7]:
encoded = gpt2_tokenizer(DEFAULT_PROMPT, return_tensors="pt")
logit_temperatures: list[float] = [0.1, 1.0, 3.0]

with torch.no_grad():
    outputs = gpt2_model(**encoded)
    logits = outputs.logits[0, -1, :]

    for logit_temperature in logit_temperatures:
        probabilities = torch.softmax(logits / logit_temperature, dim=-1)
        top_probs, top_indices = torch.topk(probabilities, 5)
        logger.info(f"Temperature = {logit_temperature:.2f}")
        for probability, token_id in zip(top_probs, top_indices):
            token = gpt2_tokenizer.decode(token_id)
            logger.info(
                f"{token!r:15} "
                f"{probability.item():.2%}")

2026-08-04 12:42:02.801 | INFO     | __main__:<module>:11 - Temperature = 0.10
2026-08-04 12:42:02.801 | INFO     | __main__:<module>:14 - ' a'            45.63%
2026-08-04 12:42:02.802 | INFO     | __main__:<module>:14 - ' the'          43.82%
2026-08-04 12:42:02.802 | INFO     | __main__:<module>:14 - ' to'           10.16%
2026-08-04 12:42:02.802 | INFO     | __main__:<module>:14 - ' that'         0.29%
2026-08-04 12:42:02.802 | INFO     | __main__:<module>:14 - ' not'          0.10%
2026-08-04 12:42:02.802 | INFO     | __main__:<module>:11 - Temperature = 1.00
2026-08-04 12:42:02.802 | INFO     | __main__:<module>:14 - ' a'            6.57%
2026-08-04 12:42:02.803 | INFO     | __main__:<module>:14 - ' the'          6.54%
2026-08-04 12:42:02.803 | INFO     | __main__:<module>:14 - ' to'           5.65%
2026-08-04 12:42:02.803 | INFO     | __main__:<module>:14 - ' that'         3.96%
2026-08-04 12:42:02.803 | INFO     | __main__:<module>:14 - ' not'          3.55%
2026-08-04 12:42:02